# Imports

In [ ]:
from copy import deepcopy

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM, LlamaModel
from tqdm import tqdm

# Load model

In [ ]:
MODEL_NAME_OR_PATH = "/home/jovyan/models/meta-llama_Meta-Llama-3.1-8B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH)

In [ ]:
tokenizer

In [ ]:
model

# Explore model embeddings layer

In [ ]:
model.model.embed_tokens

In [ ]:
state_dict = model.state_dict()

In [ ]:
state_dict["model.embed_tokens.weight"].shape

In [ ]:
state_dict["lm_head.weight"].shape

In [ ]:
[key for key in state_dict.keys() if "embed" in key]

In [ ]:
[key for key in state_dict.keys() if "lm_head" in key]

# Define added tokens

## Load new tokenizer

In [ ]:
tokenizer.vocab_size

In [ ]:
NEW_TOKENIZER_PATH = "/home/jovyan/tokenizers/hf_128000_gs3.5_latex_spaces_v12_gpt4o_rx_code_added_tokens"

In [ ]:
# NEW_TOKENIZER_PATH = "/home/jovyan/tokenizers/llama_tokenizers/extended_cc_10k"

In [ ]:
tokenizer_new = AutoTokenizer.from_pretrained(NEW_TOKENIZER_PATH)

In [ ]:
tokenizer_new

In [ ]:
tokenizer_new.vocab_size

---

In [ ]:
tokenizer.decoder, tokenizer_new.decoder

In [ ]:
tokenizer.backend_tokenizer.pre_tokenizer, tokenizer_new.backend_tokenizer.pre_tokenizer

In [ ]:
tokenizer.backend_tokenizer.normalizer, tokenizer_new.backend_tokenizer.normalizer

In [ ]:
tokenizer.backend_tokenizer.post_processor, tokenizer_new.backend_tokenizer.post_processor

In [ ]:
tokenizer.backend_tokenizer.model, tokenizer_new.backend_tokenizer.model

In [ ]:
# tokenizer_vocab_text_tokens = [
#     tokenizer.decode([token_id], skip_special_tokens=True, clean_up_tokenization_spaces=False)
#     for token, token_id in sorted(
#         tokenizer.vocab.items(), key=lambda item: item[1]
#     )
# ]
# tokenizer_new_vocab_text_tokens = [
#     tokenizer_new.decode([token_id], skip_special_tokens=True, clean_up_tokenization_spaces=False)
#     for token, token_id in sorted(
#         tokenizer_new.vocab.items(), key=lambda item: item[1]
#     )
# ]

In [ ]:
tokenizer_vocab_tokens = [
    token
    for token, _ in sorted(
        tokenizer.vocab.items(), key=lambda item: item[1]
    )
]
tokenizer_new_vocab_tokens = [
    token
    for token, _ in sorted(
        tokenizer_new.vocab.items(), key=lambda item: item[1]
    )
]

In [ ]:
tokenizer_vocab_text_tokens = [
    tokenizer.decoder.decode([token])
    for token, token_id in sorted(
        tokenizer.vocab.items(), key=lambda item: item[1]
    )
]
tokenizer_new_vocab_text_tokens = [
    tokenizer_new.decoder.decode([token])
    for token, token_id in sorted(
        tokenizer_new.vocab.items(), key=lambda item: item[1]
    )
]

In [ ]:
tokenizer_vocab_tokens[1234], tokenizer_vocab_text_tokens[1234]

In [ ]:
'under' in tokenizer_vocab_tokens

In [ ]:
'under' in tokenizer_vocab_text_tokens

In [ ]:
tokenizer.decoder.decode(['Ġunder'])

In [ ]:
tokenizer(' under', add_special_tokens=False)["input_ids"]

In [ ]:
tokenizer("    ", add_special_tokens=False)["input_ids"]

In [ ]:
tokenizer_new_vocab_tokens[128], tokenizer_new_vocab_text_tokens[128]

In [ ]:
tokenizer_new_vocab_tokens[256], tokenizer_new_vocab_text_tokens[256]

In [ ]:
all(tokenizer_new_vocab_tokens[i] == tokenizer_new_vocab_text_tokens[i] for i in range(len(tokenizer_new_vocab_text_tokens)))

In [ ]:
inp_ids = tokenizer(tokenizer_new_vocab_tokens[111], add_special_tokens=False)["input_ids"]
inp_ids

In [ ]:
tokenizer.batch_decode(inp_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

In [ ]:
tokenizer.batch_decode([112, 116, 128], skip_special_tokens=True, clean_up_tokenization_spaces=False)

In [ ]:
tokenizer_new.decoder.decode([tokenizer_new_vocab_tokens[111]])

In [ ]:
type(tokenizer_new_vocab_tokens[0])

In [ ]:
inp_ids = tokenizer("Test string with русские буквы 129879 789 7 8", add_special_tokens=False)["input_ids"]

In [ ]:
tokenizer.batch_decode(inp_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

In [ ]:
inp_ids = tokenizer_new("Test string with русские буквы 129879 789 7 8", add_special_tokens=False)["input_ids"]

In [ ]:
tokenizer_new.batch_decode(inp_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

## Determing added and intersecting tokens

In [ ]:
tokenizer_new_vocab_set = set(tokenizer_new.vocab.keys())
tokenizer_vocab_set = set(tokenizer.vocab.keys())

# tokenizer_new_vocab_set = set(tokenizer_new_vocab_text_tokens)
# tokenizer_vocab_set = set(tokenizer_vocab_text_tokens)

In [ ]:
len(tokenizer.vocab), len(tokenizer_vocab_set)

In [ ]:
len(tokenizer_new.vocab), len(tokenizer_new_vocab_set)

---

In [ ]:
tokenizer_new_repeated_toks = {}
for token in tokenizer_new_vocab_text_tokens:
    if token in tokenizer_new_repeated_toks:
        tokenizer_new_repeated_toks[token] += 1
    else:
        tokenizer_new_repeated_toks[token] = 1

In [ ]:
sorted(tokenizer_new_repeated_toks.items(), key=lambda item: item[1], reverse=True)[:20]

In [ ]:
tokenizer_repeated_toks = {}
for token in tokenizer_vocab_text_tokens:
    if token in tokenizer_repeated_toks:
        tokenizer_repeated_toks[token] += 1
    else:
        tokenizer_repeated_toks[token] = 1

In [ ]:
sorted(tokenizer_repeated_toks.items(), key=lambda item: item[1], reverse=True)[:15]

---

In [ ]:
added_tokens = list(tokenizer_new_vocab_set.difference(tokenizer_vocab_set))

In [ ]:
len(added_tokens)

In [ ]:
intersecting_tokens = list(tokenizer_new_vocab_set.intersection(tokenizer_vocab_set))

In [ ]:
len(intersecting_tokens)

In [ ]:
len(intersecting_tokens) + len(added_tokens), len(tokenizer_new.vocab)

In [ ]:
assert len(intersecting_tokens) + len(added_tokens) == len(tokenizer_new.vocab)

## Gather added and intersecting tokens with their indices in new tokenizer

In [ ]:
%%time
# added_tokens_idx = [tokenizer_new.vocab[token] for token in added_tokens]
added_tokens_idx = []
for token in tqdm(added_tokens):
    added_tokens_idx.append(tokenizer_new.convert_tokens_to_ids(token))

In [ ]:
added_tokens_idx[:5]

In [ ]:
added_tokens_with_idx = list(zip(added_tokens, added_tokens_idx))
added_tokens_with_idx = sorted(added_tokens_with_idx, key=lambda x: x[1])

In [ ]:
added_tokens_with_idx[:5]

In [ ]:
added_tokens_sorted = [token for token, _ in added_tokens_with_idx]

In [ ]:
%%time
# intersecting_tokens_idx = [tokenizer_new.vocab[token] for token in intersecting_tokens]
intersecting_tokens_idx = []
for token in tqdm(intersecting_tokens):
    # intersecting_tokens_idx.append(tokenizer_new.vocab[token])
    intersecting_tokens_idx.append(tokenizer_new.convert_tokens_to_ids(token))

In [ ]:
intersecting_tokens_with_idx = list(zip(intersecting_tokens, intersecting_tokens_idx))
intersecting_tokens_with_idx = sorted(intersecting_tokens_with_idx, key=lambda x: x[1])

In [ ]:
intersecting_tokens_with_idx[:3]

In [ ]:
assert len(set(added_tokens_idx).intersection(set(intersecting_tokens_idx))) == 0

## Get added tokens text representation

In [ ]:
added_tokens_text_repr = [
    tokenizer_new.decoder.decode([token])
    for token in added_tokens
]

In [ ]:
added_tokens_text_repr[:10], added_tokens[:10]

In [ ]:
len(set(added_tokens)), len(set(added_tokens_text_repr))

### Explore repeated tokens

In [ ]:
added_tokens_text_repr_with_ids = list(zip(added_tokens_text_repr, added_tokens_idx))
added_tokens_text_repr_with_ids = sorted(added_tokens_text_repr_with_ids, key=lambda x: x[1])

In [ ]:
added_tokens_text_repr_with_ids[:5]

In [ ]:
added_tokens_text_repr_count = {}
for token in added_tokens_text_repr:
    if token in added_tokens_text_repr_count:
        added_tokens_text_repr_count[token] += 1
    else:
        added_tokens_text_repr_count[token] = 1

In [ ]:
repeated_toks = sorted(added_tokens_text_repr_count.items(), key=lambda item: item[1], reverse=True)[:15]

In [ ]:
repeated_toks

In [ ]:
repeated_tok_text_pairs = [
    (tok, tok_text)
    for tok, tok_text in zip(added_tokens, added_tokens_text_repr)
    if tok_text == repeated_toks[0][0]
]

In [ ]:
repeated_tok_text_pairs

In [ ]:
tokenizer.tokenize(repeated_tok_text_pairs[0][0]), tokenizer.tokenize(repeated_tok_text_pairs[0][1])

In [ ]:
tokenizer.tokenize(repeated_tok_text_pairs[1][0]), tokenizer.tokenize(repeated_tok_text_pairs[1][1])

In [ ]:
tokenizer.tokenize(repeated_tok_text_pairs[2][0]), tokenizer.tokenize(repeated_tok_text_pairs[2][1])

# Calculate tokens embeddings

- Tokenizer each token using original model tokenizer
- Calculate token embedding as a mean of tokenized token embeddings

In [ ]:
model.eval()

## Input embedding layer (embed_tokens)

In [ ]:
def get_token_embedding(token: str, embedding_matrix: torch.Tensor, tokenizer: LlamaTokenizer):
    token_tokenized = tokenizer(token, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    token_embedding = embedding_matrix[token_tokenized].mean(axis=0)

    return token_embedding

In [ ]:
embed_tokens_weights = state_dict["model.embed_tokens.weight"]
hidden_size = embed_tokens_weights.shape[-1]

In [ ]:
new_embed_tokens_weights = torch.empty(len(tokenizer_new.vocab), hidden_size, dtype=embed_tokens_weights.dtype)
torch.nn.init.xavier_normal_(new_embed_tokens_weights)

In [ ]:
for token, idx in tqdm(intersecting_tokens_with_idx, total=len(intersecting_tokens_with_idx), desc="Intersecting tokens"):
    # new_embed_tokens_weights[idx] = embed_tokens_weights[tokenizer.vocab[token]]
    new_embed_tokens_weights[idx] = embed_tokens_weights[tokenizer.convert_tokens_to_ids(token)]

In [ ]:
for token, idx in tqdm(added_tokens_text_repr_with_ids, desc="Added tokens"):
    new_embed_tokens_weights[idx] = get_token_embedding(token, embed_tokens_weights, tokenizer)

In [ ]:
new_embed_tokens_weights

In [ ]:
new_embed_tokens_weights.shape

## Output embedding layer (lm_head)

In [ ]:
lm_head_weights = state_dict["lm_head.weight"]
hidden_size = lm_head_weights.shape[-1]

In [ ]:
lm_head_weights.shape

In [ ]:
new_lm_head_weights = torch.empty(len(tokenizer_new.vocab), hidden_size, dtype=lm_head_weights.dtype)
torch.nn.init.xavier_normal_(new_lm_head_weights)

In [ ]:
for token, idx in tqdm(intersecting_tokens_with_idx, desc="Intersecting tokens"):
    # new_lm_head_weights[idx] = lm_head_weights[tokenizer.vocab[token]]
    new_lm_head_weights[idx] = lm_head_weights[tokenizer.convert_tokens_to_ids(token)]

In [ ]:
for token, idx in tqdm(added_tokens_text_repr_with_ids, desc="Added tokens"):
    new_lm_head_weights[idx] = get_token_embedding(token, lm_head_weights, tokenizer)

In [ ]:
new_lm_head_weights

In [ ]:
new_lm_head_weights.shape

# Update and save model

- Reinitialize embedding and lm head layers with new state dicts
- Update model config (vocab_size)

In [ ]:
model.config

## Update state_dict weights

Use `deepcopy` as `state_dict` returns shallow copy with the references inside.

In [ ]:
embedder_state_dict = deepcopy(model.model.embed_tokens.state_dict())

In [ ]:
embedder_state_dict["weight"] = new_embed_tokens_weights

In [ ]:
embedder_state_dict["weight"].shape

In [ ]:
lm_head_state_dict = deepcopy(model.lm_head.state_dict())
lm_head_state_dict["weight"] = new_lm_head_weights

In [ ]:
lm_head_state_dict["weight"].shape

## Initialize new input embedding layer

In [ ]:
vocab_size_new = embedder_state_dict["weight"].shape[0]

In [ ]:
vocab_size_new, hidden_size

In [ ]:
model.model.embed_tokens

In [ ]:
model.model.embed_tokens.__dict__

In [ ]:
embedder_new = torch.nn.Embedding(vocab_size_new, hidden_size, padding_idx=model.model.embed_tokens.padding_idx)
embedder_new.training = model.model.embed_tokens.training

In [ ]:
embedder_new.load_state_dict(embedder_state_dict)

In [ ]:
embedder_new.__dict__

## Initialize new lm head layer

In [ ]:
model.lm_head

In [ ]:
model.lm_head.__dict__

In [ ]:
lm_head_new = torch.nn.Linear(in_features=hidden_size, out_features=vocab_size_new, bias=False)
lm_head_new.training = model.lm_head.training

In [ ]:
lm_head_new.load_state_dict(lm_head_state_dict)

In [ ]:
lm_head_new.__dict__

## Assign new layers and update config

In [ ]:
model.model.embed_tokens = embedder_new
model.lm_head = lm_head_new

In [ ]:
model.config.vocab_size = vocab_size_new

In [ ]:
model.config

In [ ]:
tokenizer_new.eos_token_id

In [ ]:
tokenizer_new.bos_token_id

In [ ]:
tokenizer_new.bos_token_id

# Save updated model

In [ ]:
SAVE_PATH = "/home/jovyan/models/meta-llama-Meta-Llama-3.1-8B-128k_tokenizer_remapped"

In [ ]:
model.save_pretrained(SAVE_PATH)

In [ ]:
tokenizer_new.save_pretrained(SAVE_PATH)

# Load and test extended model

In [ ]:
model_extended = AutoModelForCausalLM.from_pretrained(SAVE_PATH)

In [ ]:
tokenizer_extended = AutoTokenizer.from_pretrained(SAVE_PATH)

In [ ]:
model_extended

In [ ]:
tokenizer_extended

In [ ]:
test_text = "Мой тестовый текст. Вот мой домик, а вот и река. Ещё немного цифр: 2*2=4, 324324 чётное число, а 123321 трехзначное. Делу время "

In [ ]:
test_text_tokenized = tokenizer_extended(test_text, return_tensors="pt")["input_ids"]

In [ ]:
test_text_tokenized

In [ ]:
[tokenizer_extended.decode(token_id) for token_id in test_text_tokenized[0]]

In [ ]:
test_model_out = model_extended(test_text_tokenized)

In [ ]:
test_model_out.logits.shape

In [ ]:
test_text_tokenized.shape

---

In [ ]:
generate_out = model.generate(test_text_tokenized, max_new_tokens=50)

In [ ]:
generate_out

In [ ]:
generate_out.shape

In [ ]:
tokenizer_extended.decode(generate_out[0])